<a href="https://colab.research.google.com/github/Martin2513/fives-segmentation-unet-poc/blob/main/unet_poc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import zipfile
from google.colab import drive
from tqdm.auto import tqdm

import random
import numpy as np
import torch
import cv2
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [2]:
drive.mount("/content/drive")

ZIP_PATH = "/content/drive/MyDrive/FIVES.zip"
DATA_DIR = "data"

if not os.path.exists(DATA_DIR):
  with zipfile.ZipFile(ZIP_PATH, "r") as z:
    for file in tqdm(z.infolist(), desc="Extracting FIVES"):
      z.extract(file, DATA_DIR)

  print("Extracted to directory:", DATA_DIR)

Mounted at /content/drive


Extracting FIVES:   0%|          | 0/1608 [00:00<?, ?it/s]

Extracted to directory: data


In [4]:
IMG_SIZE = 768
RESIZED_DIR = "data_768"

# Find the root folder in "data"
#(where train/Original and test/Original are)
DATA_ROOT = None
for root, dirs, files in os.walk(DATA_DIR):
    if os.path.isdir(os.path.join(root, "train", "Original")) and os.path.isdir(os.path.join(root, "test", "Original")):
        DATA_ROOT = root
        break

assert DATA_ROOT is not None, "couldn't find the folders train/Original and test/Original"
print("Dataset found in:", DATA_ROOT)

# Resize and save one split ("train" or "test")
def resize_and_cache(split_name):
    src_img_dir = os.path.join(DATA_ROOT, split_name, "Original")
    src_msk_dir = os.path.join(DATA_ROOT, split_name, "Ground truth")
    dst_img_dir = os.path.join(RESIZED_DIR, split_name, "Original")
    dst_msk_dir = os.path.join(RESIZED_DIR, split_name, "Ground truth")

    os.makedirs(dst_img_dir, exist_ok=True)
    os.makedirs(dst_msk_dir, exist_ok=True)

    files = sorted([f for f in os.listdir(src_img_dir) if f.lower().endswith(".png")])
    for fname in tqdm(files, desc=f"Processing {split_name}"):
        dst_img = os.path.join(dst_img_dir, fname)
        dst_msk = os.path.join(dst_msk_dir, fname)
        if os.path.exists(dst_img) and os.path.exists(dst_msk):
            continue

        img = cv2.imread(os.path.join(src_img_dir, fname), cv2.IMREAD_COLOR)
        msk = cv2.imread(os.path.join(src_msk_dir, fname), cv2.IMREAD_GRAYSCALE)

        img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        msk_resized = cv2.resize(msk, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
        msk_resized = ((msk_resized > 127).astype("uint8")) * 255

        cv2.imwrite(dst_img, img_resized)
        cv2.imwrite(dst_msk, msk_resized)

    return len(files)


n_train = resize_and_cache("train")
n_test = resize_and_cache("test")
print(f"Saved {n_train} train and {n_test} test images in '{RESIZED_DIR}'.")

Dataset found in: data


Processing train:   0%|          | 0/600 [00:00<?, ?it/s]

Processing test:   0%|          | 0/200 [00:00<?, ?it/s]

Saved 600 train and 200 test images in 'data_768'.


In [5]:
# Set random seed for reproducibility
SEED = 881

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Helper function to extract clinical category from filename
def get_clinical_category(filename):
    base_name = os.path.splitext(filename)[0]
    clinical_category = base_name.split("_")[-1]
    return clinical_category

class FivesDataset(Dataset):
    def __init__(self, img_dir, msk_dir, filenames, augment=False):
        self.img_dir = img_dir
        self.msk_dir = msk_dir
        self.filenames = filenames
        self.augment = augment

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, index):
        fname = self.filenames[index]
        img = cv2.imread(os.path.join(self.img_dir, fname), cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        msk = cv2.imread(os.path.join(self.msk_dir, fname), cv2.IMREAD_GRAYSCALE)

        # Apply identical spatial augmentations to both image and mask
        if self.augment:
            if random.random() < 0.5:
                img = np.fliplr(img).copy()
                msk = np.fliplr(msk).copy()
            if random.random() < 0.5:
                img = np.flipud(img).copy()
                msk = np.flipud(msk).copy()
            k = random.randint(0, 3)
            if k > 0:
                img = np.rot90(img, k).copy()
                msk = np.rot90(msk, k).copy()

        # Normalize image to [0, 1] and convert to PyTorch tensor (C, H, W)
        img_tensor = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        msk_tensor = torch.from_numpy((msk > 127).astype(np.float32)).unsqueeze(0)

        return img_tensor, msk_tensor, fname


# Stratified 90/10 split of the 600 training images into train (540) and validation (60)
train_img_dir = os.path.join(RESIZED_DIR, "train", "Original")
train_msk_dir = os.path.join(RESIZED_DIR, "train", "Ground truth")
all_train_files = sorted(os.listdir(train_img_dir))
categories = [get_clinical_category(f) for f in all_train_files]

train_files, validation_files = train_test_split(all_train_files, test_size=0.1, random_state=SEED, stratify=categories)

test_img_dir = os.path.join(RESIZED_DIR, "test", "Original")
test_msk_dir = os.path.join(RESIZED_DIR, "test", "Ground truth")
test_files = sorted(os.listdir(test_img_dir))

BATCH_SIZE = 4
train_ds = FivesDataset(train_img_dir, train_msk_dir, train_files, augment=True)
validation_ds = FivesDataset(train_img_dir, train_msk_dir, validation_files, augment=False)
test_ds = FivesDataset(test_img_dir, test_msk_dir, test_files, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
validation_loader = DataLoader(validation_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)} | Validation: {len(validation_ds)} | Test: {len(test_ds)}")

Train: 540 | Validation: 60 | Test: 200
